# 04｜O2O 加算收益与持有段计数分析

本 Notebook 的输入是 **03 Notebook 已生成的含持有期精简八列表**，不重新生成八列。

- 每行是实际执行日：前一实际交易日收盘形成，当前执行日开盘可执行；
- 收益使用执行日开盘到下一实际交易日开盘的 O2O；
- 曲线使用 `1 + cumsum(持仓 × O2O)` 的加算口径，不做复利；
- 最新行如果还没有下一实际交易日开盘价，保留信号和审计行，但不进入收益评价，不填 0；
- 本 Notebook 负责收益、净值、风险和持有段/段计数；逐年拆解及 2026 年原因分析由 05 完成。

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'src' / 'generate_compact_output.py').is_file()
)
SRC_ROOT = PACKAGE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

SPOT_TEXT = os.environ.get('COMPANY_SPOT_PATH', '').strip()
if not SPOT_TEXT or not Path(SPOT_TEXT).expanduser().is_absolute():
    raise RuntimeError('请设置 COMPANY_SPOT_PATH 为本地米筐现货的绝对路径。')
SPOT_PATH = Path(SPOT_TEXT).expanduser().resolve()
HOLDING_PATH = Path(os.environ.get(
    'HOLDING_EIGHT_PATH',
    str(PACKAGE_ROOT / 'runtime_outputs_holding_period' / '含持有期八列表.csv'),
)).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get(
    'ANALYSIS_04_OUTPUT_DIR',
    str(PACKAGE_ROOT / 'runtime_outputs_04_returns'),
)).expanduser().resolve()
if not HOLDING_PATH.is_absolute() or not OUTPUT_DIR.is_absolute():
    raise RuntimeError('04 的八列表路径和输出目录都必须是绝对路径。')

from reproduce_remote_o2o import run_stage_04

print('冻结包目录：', PACKAGE_ROOT)
print('本地现货：', SPOT_PATH)
print('03 八列表：', HOLDING_PATH)
print('04 输出：', OUTPUT_DIR)

## 1. 运行 O2O 加算收益与持有段分析

In [ ]:
metadata = run_stage_04(SPOT_PATH, HOLDING_PATH, OUTPUT_DIR)
risk = pd.read_csv(OUTPUT_DIR / 'O2O加算风险指标.csv', encoding='utf-8-sig')
segment_counts = pd.read_csv(OUTPUT_DIR / '持有段计数_按系列.csv', encoding='utf-8-sig')
daily = pd.read_csv(OUTPUT_DIR / 'O2O加算逐日收益与状态.csv', encoding='utf-8-sig', parse_dates=['实际执行日', '推定形成日'])
display(risk)
display(segment_counts)
display(daily.tail(10))
print('生成文件数：', len(metadata['generated_files']))

## 2. 日期、收益和最新行检查

这里明确检查：实际执行日行的收益只取当前执行日开盘到下一实际交易日开盘；最新行没有下一开盘时只能是未评价状态，不能被补成占位收益。

In [ ]:
if not daily['形成日早于执行日'].all():
    raise AssertionError('发现形成日不早于执行日的行')
valid = daily['O2O可评价']
expected_o2o = daily.loc[valid, '下一交易日开盘'] / daily.loc[valid, '执行日开盘'] - 1.0
actual_o2o = daily.loc[valid, '执行日O2O']
if (expected_o2o - actual_o2o).abs().max() > 1e-12:
    raise AssertionError('O2O 不是执行日开盘到下一实际交易日开盘')
if daily['实际执行日'].duplicated().any():
    raise AssertionError('实际执行日重复')
latest = daily.iloc[-1]
print('最新形成日：', latest['推定形成日'])
print('最新执行日：', latest['实际执行日'])
print('最新行有下一开盘价：', bool(latest['O2O可评价']))
print('可评价行数：', int(valid.sum()), '/', len(daily))
print('检查通过：没有用占位值补齐最新收益。')

## 3. 输出位置

04 会输出 `O2O加算逐日收益与状态.csv`、`持有段明细.csv`、`持有段计数_按系列.csv`、风险指标和四张收益曲线。05 Notebook 只读取这些 04 结果做逐年与 2026 分析。